In [1]:
!pip install harmonypy=='0.0.9'

  Attempting uninstall: harmonypy
    Found existing installation: harmonypy 0.0.5
    Uninstalling harmonypy-0.0.5:
      Successfully uninstalled harmonypy-0.0.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sam-algorithm 1.0.0 requires h5py<=2.10.0, but you have h5py 3.8.0 which is incompatible.


In [2]:
!pip install anndata=='0.8.0'

In [1]:
from samalg import SAM
import scanpy as sc
import statistics
import matplotlib.pyplot as plt
import seaborn as sns
import random
import pandas as pd
import matplotlib.colors
import scipy
import numpy as np
import sklearn.metrics as metrics
from scipy import sparse
import anndata as ad
from sklearn.manifold import TSNE

/scratch/miniconda/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
#input: combined h5ad of data
#output: SAM object

In [37]:
fn = '../../Testing_Raw_Dat_RNASEQ_Joined/Backupable/tot_dat_AC_ncbi_soupx_cleaned_03122025.h5ad'

In [38]:
dataframe = sc.read_h5ad(fn)

In [4]:
#if rerunning on something that has already been normalized in the past
dataframe.X = dataframe.obsm['Raw_X']

In [5]:
for item in dataframe.X[0,:]:
    print(item)

  (0, 0)	6.0
  (0, 2)	2.0
  (0, 7)	4.0
  (0, 17)	1.0
  (0, 21)	1.0
  (0, 26)	1.0
  (0, 32)	1.0
  (0, 35)	1.0
  (0, 41)	2.0
  (0, 44)	1.0
  (0, 53)	2.0
  (0, 55)	1.0
  (0, 57)	1.0
  (0, 58)	2.0
  (0, 59)	1.0
  (0, 60)	1.0
  (0, 66)	1.0
  (0, 69)	4.0
  (0, 76)	3.0
  (0, 84)	1.0
  (0, 86)	7.0
  (0, 87)	1.0
  (0, 114)	2.0
  (0, 122)	1.0
  (0, 124)	2.0
  :	:
  (0, 19206)	1.0
  (0, 19208)	1.0
  (0, 19213)	3.0
  (0, 19214)	1.0
  (0, 19215)	1.0
  (0, 19235)	1.0
  (0, 19248)	1.0
  (0, 19249)	5.0
  (0, 19252)	19.0
  (0, 19259)	3.0
  (0, 19262)	23.0
  (0, 19266)	1.0
  (0, 19267)	5.0
  (0, 19269)	2.0
  (0, 19271)	4.0
  (0, 19275)	5.0
  (0, 19276)	1.0
  (0, 19281)	1.0
  (0, 19287)	1.0
  (0, 19288)	9.0
  (0, 19294)	4.0
  (0, 19299)	1.0
  (0, 19303)	1.0
  (0, 19304)	1.0
  (0, 19305)	2.0


In [6]:
dataframe.X

<73955x19307 sparse matrix of type '<class 'numpy.float32'>'
	with 165887644 stored elements in Compressed Sparse Column format>

In [ ]:
#Make sure gene names are in BLAST Table

In [7]:
mapping = pd.read_csv('../../BLASTMAPPING/maps/active_maps/hypo_proj/drmg/mg_to_dr.txt', delimiter = '\t', header = None)

In [8]:
gene_set = set(mapping[1])

In [9]:
a = 0
for item in dataframe.var_names:
    if item in gene_set:
        a += 1
a 

21541

In [ ]:
#Check whether the data is "raw" (UMI counts)

In [10]:
print(dataframe.X[10,:])

  (0, 6)	3.0
  (0, 5238)	2.0
  (0, 5237)	1.0
  (0, 1)	3.0
  (0, 4)	17.0
  (0, 3)	2.0
  (0, 5236)	2.0
  (0, 5235)	1.0
  (0, 14666)	2.0
  (0, 18396)	1.0
  (0, 12076)	1.0
  (0, 13315)	1.0
  (0, 18180)	1.0
  (0, 19074)	1.0
  (0, 10446)	1.0
  (0, 15045)	1.0
  (0, 8786)	1.0
  (0, 521)	1.0
  (0, 18416)	3.0
  (0, 6909)	1.0
  (0, 5473)	1.0
  (0, 22535)	1.0
  (0, 23698)	1.0
  (0, 14074)	1.0
  (0, 6787)	1.0
  :	:
  (0, 9678)	1.0
  (0, 11295)	1.0
  (0, 24433)	1.0
  (0, 24246)	1.0
  (0, 17512)	4.0
  (0, 25218)	1.0
  (0, 24656)	1.0
  (0, 8406)	1.0
  (0, 17192)	2.0
  (0, 15189)	1.0
  (0, 14271)	1.0
  (0, 21613)	1.0
  (0, 9094)	1.0
  (0, 18492)	4.0
  (0, 8090)	1.0
  (0, 23661)	1.0
  (0, 22846)	1.0
  (0, 13707)	1.0
  (0, 23256)	3.0
  (0, 25428)	1.0
  (0, 8566)	1.0
  (0, 11177)	1.0
  (0, 8842)	2.0
  (0, 13790)	1.0
  (0, 18413)	5.0


In [ ]:
#Check number of counts/genes

In [11]:
statistics.median(dataframe.obs['n_genes'])

800.0

In [ ]:
#plotting n_genes

In [ ]:
from matplotlib import rcParams

plt.rcParams['figure.figsize'] = 20,10
sns.reset_orig()
sns.violinplot(x = 'key', y = 'n_genes', data = dataframe.obs, color = 'white')
sns.stripplot(x = 'key', y = 'n_genes', data = dataframe.obs, color = 'black', size = 1)
plt.xticks(rotation=45, ha = 'right')
plt.ylim((0,6000))
plt.xlabel(None)
plt.tight_layout()
plt.savefig(fn + '_ngenes.png')
plt.show()

In [ ]:
#plotting n counts

In [ ]:
from matplotlib import rcParams

rcParams['figure.figsize'] = 20,10
sns.reset_orig()
sns.violinplot(x = 'key', y = 'n_counts', data = dataframe.obs, color = 'white')
sns.stripplot(x = 'key', y = 'n_counts', data = dataframe.obs, color = 'black', size = 1)
plt.xticks(rotation=45, ha = 'right')
plt.ylim((0,22000))
plt.xlabel(None)
plt.tight_layout()
plt.savefig(fn + '_ncounts.png')
plt.show()

In [ ]:
#extra make unique

In [7]:
dataframe.obs_names_make_unique()
dataframe.var_names_make_unique()

In [56]:
#subset dataframe (only if needed)

In [39]:
sam=SAM(dataframe)
sam.preprocess_data() #log transforms and filters the data
sam.run(batch_key='key') #run with default parameters

RUNNING SAM
Iteration: 0, Convergence: 1.0


2026-07-28 20:58:22,796 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
2026-07-28 20:58:40,506 - harmonypy - INFO - sklearn.KMeans initialization complete.


Iteration: 1, Convergence: 0.8235482900321152


2026-07-28 20:59:50,911 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
2026-07-28 21:00:12,656 - harmonypy - INFO - sklearn.KMeans initialization complete.


Iteration: 2, Convergence: 0.013408614887082558


2026-07-28 21:01:37,640 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
2026-07-28 21:01:54,583 - harmonypy - INFO - sklearn.KMeans initialization complete.


Computing the UMAP embedding...
Elapsed time: 336.1641023159027 seconds


In [ ]:
#save raw counts in obsm

In [9]:
#sam.adata.obsm['Raw_X'] = dataframe.X

In [18]:
sam.save_anndata('../../Active_SAM_joined/SAM_AC_ncbi_soupx_cleaned_03122025.h5ad')